# RQ3 — Statistical Tests: Resolver

This notebook runs the formal statistical tests supporting the RQ3 results:

1. **Wilson CIs** for the agent self-resolution rate per coding agent
2. **Chi-squared test of independence**: resolver × agent (Table 2 in paper)
3. **Fisher's exact test** for agent-assisted cells (small counts)
4. **Chi-squared test of independence**: strategy × resolver
5. **Cramér's V** effect size for strategy × resolver
6. **Wilson CIs** for per-strategy proportions within each resolver type
7. **Imprecise bias sensitivity analysis**: upper- and lower-bound estimates
   treating all Imprecise chunks as V1 (upper) or NC (lower) to bracket the
   agent vs. human contrast.

Required packages: `scipy`, `statsmodels`.  Both are in the project venv.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
for candidate in [_here, *_here.parents]:
    if (candidate / 'analysis' / 'common.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.proportion import proportion_confint

from analysis.common import (
    load_tables, build_chunk_frame, build_merge_frame,
    STRATEGY_ORDER, _RAW_TO_CANONICAL,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print('Imports OK')

## 1. Load data

In [ ]:
tables = load_tables()
chunks = build_chunk_frame(tables)
merges = build_merge_frame(tables)

chunks['strategy_canon'] = chunks['strategy'].map(_RAW_TO_CANONICAL).fillna('Imprecise')

STRATEGIES_CLASSIFIABLE = ['V1', 'V2', 'CC', 'CB', 'NC', 'NN']
classifiable = chunks[chunks['strategy_canon'] != 'Imprecise'].copy()

print(f'Total chunks:   {len(chunks):,}')
print(f'Classifiable:   {len(classifiable):,}')
print(f'Total merges:   {len(merges):,}')
print()

# Resolver column — expected values: 'agent', 'human', 'agent-assisted'
# Adjust the column name below if it differs in build_merge_frame output
RESOLVER_COL = 'resolver_type'   # merge-level resolver label
print('Resolver distribution (merge level):')
print(merges[RESOLVER_COL].value_counts())

## 2. Wilson CIs for per-agent self-resolution rates

For each coding agent: what fraction of its internal merges were committed
by the agent itself?  Wilson (score) interval at 95% confidence.

In [ ]:
# Merge-level: flag agent-resolved merges
resolver_by_agent = (
    merges
    .assign(is_agent=(merges[RESOLVER_COL] == 'agent').astype(int))
    .groupby('agent')
    .agg(n_total=('is_agent', 'count'),
         n_agent=('is_agent', 'sum'))
    .reset_index()
)

rows = []
for _, row in resolver_by_agent.iterrows():
    lo, hi = proportion_confint(row['n_agent'], row['n_total'],
                                 alpha=0.05, method='wilson')
    rows.append({
        'agent':   row['agent'],
        'n_total': int(row['n_total']),
        'n_agent': int(row['n_agent']),
        'rate':    row['n_agent'] / row['n_total'],
        'ci_lo':   lo,
        'ci_hi':   hi,
    })

ci_df = pd.DataFrame(rows).sort_values('rate', ascending=False)
ci_df['rate_pct'] = ci_df['rate'].map('{:.1%}'.format)
ci_df['ci_95'] = ci_df.apply(lambda r: f'[{r.ci_lo:.1%}, {r.ci_hi:.1%}]', axis=1)
print(ci_df[['agent', 'n_total', 'n_agent', 'rate_pct', 'ci_95']].to_string(index=False))

## 3. Chi-squared test of independence: resolver × agent

H₀: The resolver distribution (agent / human / agent-assisted) is independent
of the coding agent that authored the PR.

This is the formal test for Table 2 in the paper.

**Cramér's V** (bias-corrected) is added as primary effect-size metric.  
This test is at merge level (one observation per merge), so clustering is less
severe than chunk-level tests. Within-repo correlation still slightly inflates
both χ² and V; treat V as an upper bound on the true effect.

In [ ]:
contingency_ra = (
    merges
    .groupby(['agent', RESOLVER_COL])
    .size()
    .unstack(fill_value=0)
)
print('Contingency table (resolver × agent):')
print(contingency_ra.to_string())

chi2_ra, p_ra, dof_ra, expected_ra = chi2_contingency(contingency_ra.values)

# Cramér's V — primary effect metric
n_merges = int(contingency_ra.values.sum())
r_ra, c_ra = contingency_ra.shape
V_ra = cramers_v(chi2_ra, n_merges, r_ra, c_ra)
label_ra = 'weak' if V_ra < 0.10 else ('moderate' if V_ra < 0.30 else 'strong')

print(f'\nChi-squared: χ²({dof_ra}) = {chi2_ra:.1f},  p = {p_ra:.2e}')
print(f"Cramér's V (bias-corrected) = {V_ra:.3f}  → {label_ra} effect  [primary metric]")
min_exp = expected_ra.min()
pct_lt5 = (expected_ra < 5).sum() / expected_ra.size * 100
print(f'Min expected count: {min_exp:.2f}  |  Cells < 5: {pct_lt5:.1f}%')
if pct_lt5 > 20:
    print('WARNING: Small cells present. Consider collapsing agent-assisted with agent.')


## 4. Wilson CIs for agent-assisted rate per coding agent

The agent-assisted bucket has very few total cases (n_assisted ≈ 54 total across
all agents). The original analysis used Fisher's exact test with a
'each agent vs all others' 2×2 framing, but this framing is invalid:
the cell sizes in the 'all others' row vary with agent size, making p-values
non-comparable across rows.

**Wilson CIs** are the correct summary: they directly quantify the uncertainty
around each agent's own assisted rate without requiring a distributional assumption.
If the CIs for an agent exclude zero (lower bound > 0), assisted resolutions are
detectable for that agent.

In [ ]:
assisted_by_agent = (
    merges
    .assign(is_assisted=(merges[RESOLVER_COL] == 'agent-assisted').astype(int))
    .groupby('agent')
    .agg(n_total=('is_assisted', 'count'),
         n_assisted=('is_assisted', 'sum'))
    .reset_index()
)

from statsmodels.stats.proportion import proportion_confint

print(f"{'Agent':<22} {'Rate':>8} {'95% CI':>22} {'n_asst/n_total':>16}")
print('-' * 72)
for _, row in assisted_by_agent.iterrows():
    n_a = int(row['n_assisted'])
    n   = int(row['n_total'])
    lo, hi = proportion_confint(n_a, n, alpha=0.05, method='wilson')
    rate = n_a / n
    print(f"{row['agent']:<22} {rate:>8.1%} [{lo:.1%}, {hi:.1%}]  {n_a}/{n}")

print()
print('NOTE: Wilson CI lower bound > 0 indicates assisted resolutions are detectable.')
print('Fisher exact was removed: the "each vs all others" 2×2 framing is asymmetric')
print('(cell sizes in the reference row vary by agent size), making p-values')
print('non-comparable across rows.')


## 5. Chi-squared test of independence: strategy × resolver

H₀: The strategy distribution (V1, V2, CC, CB, NC, NN) is the same for
agent-resolved and human-resolved chunks.

This is the core statistical test for the 86.6% vs 45.2% V1 contrast.

In [ ]:
# Join chunk-level resolver info
# Assuming resolver_type is on chunks (via build_chunk_frame join with merges)
# If not present, you may need to merge chunks with merges on (repo_full_name, merge_sha)

if RESOLVER_COL not in classifiable.columns:
    # Try to merge from merges frame
    merge_keys = ['repo_full_name', 'merge_sha']
    classifiable = classifiable.merge(
        merges[merge_keys + [RESOLVER_COL]],
        on=merge_keys, how='left'
    )
    chunks = chunks.merge(
        merges[merge_keys + [RESOLVER_COL]],
        on=merge_keys, how='left'
    )

# Restrict to agent / human (exclude agent-assisted, too few)
classif_ah = classifiable[classifiable[RESOLVER_COL].isin(['agent', 'human'])]

contingency_sr = (
    classif_ah
    .groupby([RESOLVER_COL, 'strategy_canon'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=STRATEGIES_CLASSIFIABLE, fill_value=0)
)

print('Contingency table (strategy × resolver):')
print(contingency_sr.to_string())

# Row percentages
print('\nRow percentages:')
print(contingency_sr.div(contingency_sr.sum(axis=1), axis=0).map('{:.1%}'.format).to_string())

chi2_sr, p_sr, dof_sr, exp_sr = chi2_contingency(contingency_sr.values)
print(f'\nChi-squared: χ²({dof_sr}) = {chi2_sr:.1f},  p = {p_sr:.2e}')

## 6. Cramér's V effect size (strategy × resolver)

In [ ]:
def cramers_v(chi2_val, n, r, c):
    """Bias-corrected Cramér's V (Bergsma 2013)."""
    phi2 = chi2_val / n
    phi2_corr = max(0, phi2 - (r - 1) * (c - 1) / (n - 1))
    r_corr = r - (r - 1) ** 2 / (n - 1)
    c_corr = c - (c - 1) ** 2 / (n - 1)
    return np.sqrt(phi2_corr / min(r_corr - 1, c_corr - 1))

n_sr = contingency_sr.values.sum()
r_sr, c_sr = contingency_sr.shape
V_sr = cramers_v(chi2_sr, n_sr, r_sr, c_sr)

label_sr = ('weak' if V_sr < 0.10
            else 'moderate' if V_sr < 0.30
            else 'strong')
print(f"Cramér's V (bias-corrected) = {V_sr:.3f}  → {label_sr} effect")

## 7. Wilson CIs for strategy proportions within each resolver type

This table produces the per-strategy proportions with uncertainty bounds
for both agent-resolved and human-resolved chunks.

In [ ]:
rows_ci = []
for resolver in ['agent', 'human']:
    sub = classif_ah[classif_ah[RESOLVER_COL] == resolver]
    n_sub = len(sub)
    for strat in STRATEGIES_CLASSIFIABLE:
        cnt = (sub['strategy_canon'] == strat).sum()
        lo, hi = proportion_confint(cnt, n_sub, alpha=0.05, method='wilson')
        rows_ci.append({
            'resolver': resolver,
            'strategy': strat,
            'count': cnt,
            'n': n_sub,
            'pct': f'{cnt/n_sub:.1%}',
            'ci_95': f'[{lo:.1%}, {hi:.1%}]',
        })

ci_resolver_df = pd.DataFrame(rows_ci)
print(ci_resolver_df[['resolver', 'strategy', 'count', 'n', 'pct', 'ci_95']].to_string(index=False))

## 8. Sensitivity analysis: Imprecise bias

The agent-resolved chunks have a much lower Imprecise rate (13.4%) than
human-resolved chunks (40.8%). This asymmetry could inflate the apparent
V1 dominance of agents if V1 resolutions are easier to localize.

We bracket the 86.6% agent-V1 estimate with:
  - **Upper bound**: assume all Imprecise chunks are V1 (most conservative re: agent).
  - **Lower bound**: assume all Imprecise chunks are NC (new code, hardest case).
  - **Neutral bound**: distribute Imprecise proportionally to existing distribution.

If the contrast persists even under the lower bound, the finding is robust.

In [ ]:
all_ah = chunks[chunks[RESOLVER_COL].isin(['agent', 'human'])].copy()

def compute_scenario(df, resolver, imprecise_label):
    """Compute V1 rate for a given resolver under a given Imprecise assumption."""
    sub = df[df[RESOLVER_COL] == resolver].copy()
    sub['strategy_used'] = sub['strategy_canon'].replace({'Imprecise': imprecise_label})
    counts = sub['strategy_used'].value_counts()
    n_total = len(sub)
    v1_rate = counts.get('V1', 0) / n_total
    return v1_rate, n_total

scenarios = [
    ('upper bound (Imprecise → V1)',   'V1'),
    ('neutral (Imprecise → NN)',        'NN'),   # NN is the least common, keeps V1 unchanged
    ('lower bound (Imprecise → NC)',    'NC'),
]

print(f'{'Scenario':<40} {'Agent V1%':>12} {'Human V1%':>12} {'Ratio':>8}')
print('-' * 76)
for label, imp_label in scenarios:
    a_v1, a_n = compute_scenario(all_ah, 'agent', imp_label)
    h_v1, h_n = compute_scenario(all_ah, 'human', imp_label)
    print(f'{label:<40} {a_v1:>11.1%} {h_v1:>11.1%} {a_v1/h_v1:>7.2f}x')

print()
print('Classifiable only (baseline):')
a_base = (classif_ah[classif_ah[RESOLVER_COL]=='agent']['strategy_canon']=='V1').mean()
h_base = (classif_ah[classif_ah[RESOLVER_COL]=='human']['strategy_canon']=='V1').mean()
print(f'{"classifiable only":<40} {a_base:>11.1%} {h_base:>11.1%} {a_base/h_base:>7.2f}x')

## 9. Summary

In [ ]:
print('=== RQ3 STATISTICAL TEST SUMMARY ===')
print()
print('(1) Wilson CIs for agent self-resolution rate — see ci_df')
print(f'(2) Independence resolver × agent: χ²({dof_ra}) = {chi2_ra:.1f}, p = {p_ra:.2e}')
print(f"    Cramér's V = {V_ra:.3f} ({label_ra})  [primary effect metric]")
print('(3) Wilson CIs for agent-assisted rate per agent (Fisher exact removed — invalid framing)')
print(f'(4) Independence strategy × resolver: χ²({dof_sr}) = {chi2_sr:.1f}, p = {p_sr:.2e}')
print(f"(5) Cramér's V (strategy × resolver) = {V_sr:.3f} ({label_sr} effect)")
print('(6) Wilson CIs per strategy within each resolver — see ci_resolver_df')
print('(7) Sensitivity analysis for Imprecise bias — see table above')
print()
print('VALIDITY NOTE: p-values are anti-conservative due to within-merge/within-repo')
print('clustering. Cramér\'s V (bias-corrected, Bergsma 2013) is the primary effect')
print('metric; it shares the independence assumption but is far less N-sensitive.')
print('Treat all V values as slight upper bounds due to residual clustering effects.')
